In [33]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
from IPython.display import display, Markdown
import os
from pathlib import Path
from agents.extensions.models.litellm_model import LitellmModel

load_dotenv(override=True)
print(f"OpenRouter API Key found: {bool(os.environ.get('OPENROUTER_API_KEY'))}")

OpenRouter API Key found: True


In [ ]:
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')


In [61]:
params = {
      "command": "npx",
      "args": [
        "-y",
        "mcp-remote",
        f"https://mcp.tavily.com/mcp/?tavilyApiKey={TAVILY_API_KEY}"
      ],
      "env": {
        "DEFAULT_PARAMETERS": str({"include_images": False,"include_answer": True,  "search_depth": "advanced", "max_results": 10})
      }
}

tool_filter=create_static_tool_filter(blocked_tool_names=["tavily_research","tavily_map","tavily_crawl"])

In [40]:
root_dir = Path(os.getcwd()).parent
abs_tmp_dir = str((root_dir / "tmp").resolve())

mcp_server_python = MCPServerStdio(
    name="Sandboxed Workspace",
    params={ 
        "command": "docker", 
        "args": [ 
            "run", "-i", "--rm", 
            "--security-opt", "no-new-privileges",
            "--cap-drop", "ALL",
            "--init",
            "--memory", "512m",
            "--cpus", "0.5",
            "-v", f"{abs_tmp_dir}:/workspace", 
            "ghcr.io/hrrodan/agent-workspace-mcp:latest" 
        ] 
    }, 
    client_session_timeout_seconds=60.0 
)

In [39]:
abs_tmp_dir

'/home/martin/Python/Projects/Github/agents/6_mcp/tmp'

In [55]:
instructions = "You have tools available, use them if necessary."
request = "Check the docs from functools online and write a small example script in your workspace. Use advanced search and include answers when querying the tavily api."
model = "openrouter/google/gemini-3-flash-preview"

In [62]:

async with MCPServerStdio(params=params,tool_filter=tool_filter ,client_session_timeout_seconds=30) as mcp_server:
    async with mcp_server_python as mcp_server_python:
        agent = Agent(name="tool_manager", instructions=instructions, model=LitellmModel(model), mcp_servers=[mcp_server, mcp_server_python])
        with trace("tool_manager"):
            result = await Runner.run(agent, request, max_turns=30)
        display(Markdown(result.final_output))



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



Based on the official Python documentation and common use cases, I have created a small example script demonstrating four key features of the `functools` module.

### Key `functools` features included:
1.  **`@lru_cache`**: Optimizes recursive or expensive functions by caching results.
2.  **`partial`**: Generates a new version of a function with some arguments pre-filled.
3.  **`@wraps`**: Essential for writing decorators; it ensures the original function's metadata (like `__name__` and docstrings) are preserved.
4.  **`reduce`**: Performs cumulative calculations across an iterable.

### Example Script: `functools_example.py`

```python
import functools
import time

# 1. @lru_cache: Caches results of function calls to speed up repetitive tasks
@functools.lru_cache(maxsize=128)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

# 2. partial: Creates a new function with some arguments pre-filled
def power(base, exponent):
    return base ** exponent

square = functools.partial(power, exponent=2)
cube = functools.partial(power, exponent=3)

# 3. @wraps: Preserves metadata (like __name__ and __doc__) of the wrapped function
def my_logger(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling function: {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@my_logger
def say_hello(name):
    """Greets a person by name."""
    print(f"Hello, {name}!")

# 4. reduce: Applies a rolling computation to a list
numbers = [1, 2, 3, 4, 5]
sum_result = functools.reduce(lambda x, y: x + y, numbers)

if __name__ == "__main__":
    print("--- Testing lru_cache (Fibonacci) ---")
    start = time.time()
    print(f"Fibonacci(35): {fibonacci(35)}")
    print(f"Time taken with cache: {time.time() - start:.5f}s")
    
    print("\n--- Testing partial ---")
    print(f"Square of 5: {square(5)}")
    print(f"Cube of 5: {cube(5)}")
    
    print("\n--- Testing wraps ---")
    say_hello("Alice")
    print(f"Function Name: {say_hello.__name__}")
    print(f"Docstring: {say_hello.__doc__}")
    
    print("\n--- Testing reduce ---")
    print(f"Sum of {numbers}: {sum_result}")
```

### Execution Results:
When running the script, you can see how `@lru_cache` makes the Fibonacci calculation nearly instantaneous, and how `@wraps` correctly preserves the name and docstring of `say_hello` even after being decorated.

```text
--- Testing lru_cache (Fibonacci) ---
Fibonacci(35): 9227465
Time taken with cache: 0.00005s

--- Testing partial ---
Square of 5: 25
Cube of 5: 125

--- Testing wraps ---
Calling function: say_hello
Hello, Alice!
Function Name: say_hello
Docstring: Greets a person by name.

--- Testing reduce ---
Sum of [1, 2, 3, 4, 5]: 15
```